# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanvir-Sheikh-R/From-flyrank-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — connect to the warehouse

Runs once. Iterate on `month=2026-03` (a mid-panel month) — never on `_sample`, which is the sealed final month (June 2026).


In [26]:
import duckdb, os
from dotenv import load_dotenv
load_dotenv()

HF_TOKEN = os.environ.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected. Working month: 2026-03 (mid-panel, safe for iterating).")


Connected. Working month: 2026-03 (mid-panel, safe for iterating).


## 1. Unit of analysis + time window
One row = one content page (`content_hash_id`), aggregated over March 2026 
(`month=2026-03`), joined from `fact_content_daily_performance` to `dim_content` 
for metadata. I verify this below with a grain query.

In [37]:
con.sql(f"DESCRIBE SELECT * FROM {DIM_CONTENT}").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Rows violating the grain (should be EMPTY):")
grain_check

Rows violating the grain (should be EMPTY):


,report_date,client_hash_id,content_hash_id,n


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` | Feature | Observed search performance, knowable before the decision moment |
| `word_count`, `content_age_days` (from `dim_content`) | Feature | Content metadata, fixed before the window I'm scoring |
| `report_date`, `client_hash_id`, `content_hash_id` | Context | Grouping/joining/splitting only — never fed to a model |
| Impression trend (last-30 vs prev-30 within the month) | Label / proxy | The thing I'm predicting: is this page declining? |
| `health_score`, `priority_score`, any FlyRank product flag | Excluded | Not shipped in this dataset, and even if it were: using a product's own decision as a feature just teaches the model to copy an old rule, not to find real signal |

**The label, in one sentence:** a page is `is_declining` when its last-30-day impressions within the window dropped more than 20% versus the prior 30 days — the same simple rule the starter dataset uses, built here myself from the raw daily table.


## 3. Verify it with queries (grain, counts, missing values, windows)

Grain was proved in Section 1. Two more facts to verify, then the five-feature frame, then the deliberate leakage trap.

### 3a. Size + date span


In [28]:
counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_pages,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT}
""").df()
counts


,n_rows,n_pages,n_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


### 3b. Availability check — always `IS TRUE`, never `= FALSE`

`ga4_data_available` can be `NULL`, not just `TRUE`/`FALSE` — a row before a client's GA4 tracking start has GA4 columns zero-filled, which is "not measured," not "zero engagement." Filtering `= FALSE` or `NOT flag` silently miscounts, because `NULL` is neither true nor false. `IS TRUE` is the only safe filter.


In [29]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {FACT}
""").df()

share = availability["ga4_available_rows"][0] / availability["total_rows"][0]
print(availability)
print(f"Share of rows with real GA4 data: {share:.2%}")


   total_rows  ga4_available_rows
0     9841378            413966.0
Share of rows with real GA4 data: 4.21%


### 3c. Five features — max five, each with an "available when?" line

1. **`log_impressions`** — log-scaled GSC impressions over the month. Available when: known at the end of the scoring window, before any refresh decision is made.
2. **`avg_position`** — mean GSC search position over the month. Available when: computed purely from search performance already observed, no future data needed.
3. **`days_since_last_update`** *(from `dim_content`)* — content freshness. Available when: known at any moment, it only depends on the page's own edit history.
4. **`word_count`** *(from `dim_content`)* — content length. Available when: fixed at publish/edit time, never changes based on the outcome I'm predicting.
5. **`ctr`** — clicks ÷ impressions for the month. Available when: derived only from this month's observed search activity, not from anything in the future.


In [ ]:
feature_frame = con.sql(f"""
    WITH page_month AS (
        SELECT
            f.content_hash_id,
            f.client_hash_id,
            SUM(f.gsc_impressions) AS impressions,
            SUM(f.gsc_clicks) AS clicks,
            AVG(NULLIF(f.gsc_avg_position, 0)) AS avg_position
        FROM {FACT} f
        GROUP BY 1, 2
        HAVING SUM(f.gsc_impressions) >= 100
        )
        SELECT
            pm.content_hash_id,
            pm.client_hash_id,
            LN(1 + pm.impressions) AS log_impressions,
            pm.avg_position,
            pm.clicks / NULLIF(pm.impressions, 0) AS ctr,
            d.word_count,
            DATE_DIFF('day', d.content_created_date, DATE '2026-03-31') AS content_age_days
        FROM page_month pm
        LEFT JOIN {DIM_CONTENT} d USING (content_hash_id)
    """).df()

print(f"{len(feature_frame):,} pages with enough March 2026 volume to feature-ize")
feature_frame.head()

101,441 pages with enough March 2026 volume to feature-ize


,content_hash_id,client_hash_id,log_impressions,avg_position,ctr,word_count,content_age_days
0,content_96aaf57368159497,client_e547b89c05043229,7.169350,24.283004,0.005393,2855,375
1,content_1c8fb86fa8c38842,client_e547b89c05043229,6.869014,15.773976,0.002081,2460,375
2,content_53054dbaad90f335,client_e547b89c05043229,5.252273,20.075823,0.000000,2619,375
3,content_f94d1937b8cd4ddc,client_e547b89c05043229,6.591674,23.539806,0.000000,<NA>,375
4,content_8b826a45386d1f78,client_e547b89c05043229,9.224440,4.982085,0.000789,3044,417


### 3d. The leakage trap 

Step 1: build the label (declining vs not) from last-15 vs first-15 days of the month as a stand-in for last-30/prev-30. Step 2: add ONE label-derived column as a "feature" and watch a quick model score jump toward perfect. Step 3: delete it and keep the honest number.


In [36]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

labeled = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_last_half,
        SUM(CASE WHEN report_date <  DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half
    FROM {FACT}
    GROUP BY 1
    HAVING imp_first_half >= 50
""").df()

labeled["trend_pct"] = (labeled["imp_last_half"] - labeled["imp_first_half"]) / labeled["imp_first_half"] * 100
labeled["is_declining"] = (labeled["trend_pct"] < -20).astype(int)

data = feature_frame.merge(labeled[["content_hash_id", "trend_pct", "is_declining"]], on="content_hash_id")
data = data.fillna(0)

honest_features = ["log_impressions", "avg_position", "ctr", "word_count", "content_age_days"]
X_honest = data[honest_features]
y = data["is_declining"]

tree_honest = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_honest, y)
score_honest = tree_honest.score(X_honest, y)
print(f"Honest in-sample accuracy (5 real features only): {score_honest:.3f}")

# Now the trap: feed the label-derived column in as a "feature".
X_leaky = data[honest_features + ["trend_pct"]]
tree_leaky = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_leaky, y)
score_leaky = tree_leaky.score(X_leaky, y)
print(f"'Leaky' accuracy with trend_pct included:      {score_leaky:.3f}  <- suspiciously perfect")
print()
print("trend_pct is literally what is_declining is thresholded from -> that's the leak.")
print("Deleting it and keeping the honest number above.")


Honest in-sample accuracy (5 real features only): 0.631
'Leaky' accuracy with trend_pct included:      1.000  <- suspiciously perfect

trend_pct is literally what is_declining is thresholded from -> that's the leak.
Deleting it and keeping the honest number above.


## 4. Data limits

This slice cannot tell me whether a decline is a real content problem versus seasonality, SERP changes, or a sibling page absorbing the same demand — the daily fact table alone has no way to separate those causes. It also reflects an **unbalanced panel**: some clients have over a year of history in `dim_clients.gsc_data_start`, others only a few months, so a single month like March 2026 will contain far more rows from long-tracked clients than newly onboarded ones. Any pattern I see here is observed and directional for this one month, not a guarantee that holds for every client or every month.


In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_pages,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT}
""").df()
print(counts)

    n_rows  n_pages   min_date   max_date
0  9841378   331437 2026-03-01 2026-03-31


In [39]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {FACT}
""").df()
print(availability)
print(f"Share with real GA4 data: {availability['ga4_available_rows'][0] / availability['total_rows'][0]:.2%}")

   total_rows  ga4_available_rows
0     9841378            413966.0
Share with real GA4 data: 4.21%


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.